# Week 12: Dimensionality Reduction & PCA

## Strategy: PCA-Guided Subspace Optimization
With 21 data points, we can now analyze the "Principal Components" of our search space.
1. **High-Yield Cluster Extraction:** Isolate the top performing points.
2. **PCA Analysis:** Fit `sklearn.decomposition.PCA` on this cluster to identify which dimensions carry the most variance.
3. **Dimensionality Reduction (Conceptual):** Tighten the bounds severely on "redundant" (low-variance) dimensions to exploit them, and allow wider bounds on "Principal" (high-variance) dimensions to keep exploring.

In [1]:
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.optimize import minimize
import sys
import os

# Ensure we can import from src directory
sys.path.append(os.path.abspath('..'))
from src.utils import load_data

warnings.filterwarnings("ignore")
np.random.seed(52) # Week 12 deterministic seed

print("Ready for PCA-Guided Optimization")

Ready for PCA-Guided Optimization


In [2]:
def suggest_next_point_pca(func_id, X_train, y_train):
    print(f"\n--- Optimizing Function {func_id} (PCA-Guided) ---")
    
    # 1. Preprocessing
    scaler_x = StandardScaler()
    X_scaled = scaler_x.fit_transform(X_train)
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    
    # 2. Surrogate Models (Scaled Ensemble from previous weeks)
    hidden_layers = (128, 64) if func_id in [7, 8] else (64, 32)
    base_mlp = MLPRegressor(hidden_layer_sizes=hidden_layers, alpha=0.01, 
                            activation='tanh', solver='lbfgs', max_iter=2000)
    
    regr = BaggingRegressor(estimator=base_mlp, n_estimators=20, random_state=42, n_jobs=-1)
    regr.fit(X_scaled, y_scaled)
    
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)
    gp_model = GaussianProcessRegressor(kernel=kernel, normalize_y=False)
    gp_model.fit(X_scaled, y_scaled)
    
    # 3. PCA & Clustering Logic
    k_cluster = max(4, len(y_train) // 4) # Top 25%
    top_indices = np.argsort(y_train)[-k_cluster:]
    top_X_real = X_train[top_indices]
    centroid_X_real = np.mean(top_X_real, axis=0)
    
    # Apply PCA to the High-Yield Cluster
    # Note: n_components cannot exceed min(n_samples, n_features)
    n_components = min(k_cluster, X_train.shape[1])
    pca = PCA(n_components=n_components)
    pca.fit(top_X_real)
    
    # Calculate variance per original feature based on PCA reconstruction/components
    # This shows which dimensions act as the "Principal Components" in our search space
    feature_variance = np.var(top_X_real, axis=0)
    
    print(f"   [PCA Log] Explained Variance Ratio: {pca.explained_variance_ratio_}")
    
    # Dimensionality Reduction Strategy: 
    # We scale the boundary based on the variance. If a dimension has almost 0 variance in 
    # the top cluster, it means we have "converged" on that axis. We reduce its exploration.
    dynamic_radius = np.clip(feature_variance * 3.0, 0.01, 0.20)
    
    # Calculate boundaries in scaled space
    bounds_scaled = []
    for i in range(X_train.shape[1]):
        mean_val, scale_val = scaler_x.mean_[i], scaler_x.scale_[i]
        c_val = centroid_X_real[i]
        r_val = dynamic_radius[i]
        
        lower = (max(0.0, c_val - r_val) - mean_val) / scale_val
        upper = (min(1.0, c_val + r_val) - mean_val) / scale_val
        bounds_scaled.append((lower, upper))

    # 4. Objective Function
    centroid_X_scaled = scaler_x.transform(centroid_X_real.reshape(1, -1)).flatten()
    
    def objective_function(x):
        x_reshaped = x.reshape(1, -1)
        nn_preds = np.array([est.predict(x_reshaped)[0] for est in regr.estimators_])
        avg_nn, std_nn = np.mean(nn_preds), np.std(nn_preds)
        
        gp_pred, gp_std = gp_model.predict(x_reshaped, return_std=True)
        gp_pred, gp_std = gp_pred[0], gp_std[0]
        
        comb_mean = 0.6 * avg_nn + 0.4 * gp_pred
        comb_std = 0.6 * std_nn + 0.4 * gp_std
        
        # UCB with tight PCA bounds focuses naturally on exploitation
        ucb = comb_mean + 1.96 * comb_std
        
        # Repulsion to avoid duplicating exact centroid
        dist_sq = np.sum((x_reshaped - centroid_X_scaled.reshape(1, -1))**2)
        penalty = 5.0 * np.exp(-dist_sq / (2 * 0.05**2))
        
        return -ucb + penalty

    # 5. Optimization step
    x_init = centroid_X_scaled + np.random.uniform(-0.01, 0.01, size=centroid_X_scaled.shape)
    res = minimize(fun=objective_function, x0=x_init, method='L-BFGS-B', 
                   bounds=bounds_scaled, options={'maxiter': 100})
    
    next_point = np.clip(scaler_x.inverse_transform(res.x.reshape(1, -1)).flatten(), 0.0, 1.0)
    return next_point

In [3]:
submission_queries = {}
print("Starting Round 12 Evaluation (PCA/Dimensionality Reduction)...")

for func_id in range(1, 9):
    # Update data to 21 points before running
    X_known, y_known = load_data(func_id)
    next_x = suggest_next_point_pca(func_id, X_known, y_known)
    submission_queries[func_id] = next_x

print("\n" + "="*30)
print("FORMATTED SUBMISSION OUTPUT")
print("="*30)

for func_id, x_val in submission_queries.items():
    formatted_str = "-".join([f"{val:.6f}" for val in x_val])
    print(f"function_number: {func_id}: {formatted_str}")

   [PCA Log] Explained Variance Ratio: [3.74031766e-01 3.20213603e-01 1.71760825e-01 8.77017330e-02
 2.86070861e-02 1.37383642e-02 3.59380164e-03 3.52821501e-04]

FORMATTED SUBMISSION OUTPUT
function_number: 1: 0.692128-0.616611
function_number: 2: 0.600523-0.884836
function_number: 3: 0.496443-0.517062-0.450699
function_number: 4: 0.436748-0.411282-0.415455-0.423625
function_number: 5: 0.989734-0.978105-0.989254-1.000000
function_number: 6: 0.316146-0.371666-0.547240-0.758396-0.069065
function_number: 7: 0.005817-0.305669-0.357113-0.204450-0.407816-0.719028
function_number: 8: 0.007506-0.132897-0.023310-0.098036-0.746159-0.588134-0.056415-0.729791
